In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
Cs137_MeV = 0.662
Na22_MeV = [0.511, 1.275, 1.786]
Co60_MeV = [1.173, 1.332, 2,505]
Am241_MeV = 0.0595
# Background emission
K40_MeV = 1.460
Tl208_MEV = 2.614

colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kViolet, ROOT.kYellow, ROOT.kSpring, ROOT.kCyan,]

In [2]:
files = sorted((project_root/'data/260427').glob('SUBT_*'))
acqs = [SiphraAcquisition(file) for file in files]
#print(acqs)
bg_acq = acqs[12]
na_acq = acqs[11]
cs_acq = acqs[13]
co_acq = acqs[14]

long_acqs = acqs[11:15]
print(long_acqs)

bg_acqs = [bg_acq, bg_acq, bg_acq]
src_acqs = [na_acq, cs_acq, co_acq]


[SIPHRAAcquisition(File: 'SUBT_12_SIPHRA16Test_1--100_1--3_triggerOnAll_Na22'), SIPHRAAcquisition(File: 'SUBT_13_SIPHRA16Test_1--100_1--3_triggerOnAll_BG'), SIPHRAAcquisition(File: 'SUBT_14_SIPHRA16Test_1--100_1--3_triggerOnAll_Cs137'), SIPHRAAcquisition(File: 'SUBT_15_SIPHRA16Test_1--100_1--3_triggerOnAll_Co60')]


In [3]:
# histograms
hists = {}
sources = []
for sgnl, bg in zip(src_acqs, bg_acqs):
    filepath = sgnl.filepath.stem
    src = (MetadataLoader.load(sgnl.metadataFile)).source
    print(src)
    sources.append(src)
    # Signal and Background
    hist_sgnl = ROOT.TH1F(f"{src} Signal", "", BITS12, 0, BITS12)
    hist_bg = ROOT.TH1F(f"{src} Background", "", BITS12, 0 , BITS12)
    hist_sgnl.Fill(sgnl['s']/len(sgnl.active_chs))
    hist_bg.Fill(bg['s']/len(bg.active_chs))
    hist_bg.Scale(sgnl.exposure/bg.exposure)
    # Clean spectrum
    hist_clean = hist_sgnl.Clone(f"{src} Clean")
    hist_clean.Add(hist_bg, -1)
    for hist in [hist_sgnl, hist_bg, hist_clean]:
        # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Normalized counts")
    hists[src] = hist_sgnl
    hists[f"{src}_BG"] = hist_bg
    hists[f"{src}_clean"] = hist_clean
    del hist_sgnl, hist_bg

Na-22
Cs-137
Co-60


In [4]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1600, 1200)
cv.Divide(2,2)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for idx, (src, lg) in enumerate(zip(sources, lgds)):   
    cv.cd(idx+1)
    
    sgnl = hists[src]
    bg = hists[src+'_BG']
    clean = hists[src+'_clean']
    
    lg.AddEntry(sgnl, "Signal")
    lg.AddEntry(bg, "Background")
    lg.AddEntry(clean, "Subtracted")
    sgnl.SetLineColor(colors[0])
    bg.SetLineColor(colors[1])
    clean.SetLineColor(colors[2])
    sgnl.SetTitle(src)
    sgnl.Draw("hist")
    bg.Draw("sames hist")
    clean.Draw("sames hist")
    ROOT.gPad.Update()
    for i, sp in enumerate([sgnl, bg, clean]):
        st = sp.FindObject("stats")
        # print(type(st))
        st.SetY1NDC(Yinit - i*0.08)
        st.SetY2NDC(0.1)
    lg.Draw()
    cv.cd(idx+1).SetLogy()
    cv.Draw()

In [5]:
from analysis import *

#Calibration based on the peak of the Cs-137 spectrum


hist = hists['Cs-137_clean']
energy_ranges = [(0,1), (100, 250)]
energies = [0, Cs137_MeV]

c_fit = calibration_fit(hist, energy_ranges, energies)
print(c_fit)

[0.005047552967349819, -0.0017249559334720239]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  4.53122e-07
NDf                       =            0
Edm                       =  4.52719e-07
NCalls                    =           65
Constant                  =      44.4141   +/-   156.412     
Mean                      =     0.341741   +/-   1.03107     
Sigma                     =     0.223833   +/-   1.59356      	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      966.979
NDf                       =          147
Edm                       =  4.78266e-08
NCalls                    =           75
Constant                  =      9836.75   +/-   19.778      
Mean                      =      131.494   +/-   0.0326567   
Sigma                     =      17.0284   +/-   0.0262917    	 (limited)
****************************************
Minimizer is Linear / Migrad
Chi2             

In [6]:
#Calibrating Cs137 histograms based on Na22 calibration fit
hist_cal_bg_Cs137 = calibrated_histogram(c_fit, acqs[2], BITS12)
hist_cal_bg_Cs137.Scale(1/acqs[2].exposure) 
hist_cal_src_Cs137 = calibrated_histogram(c_fit, acqs[1], BITS12)
hist_cal_src_Cs137.Scale(1/acqs[1].exposure) 
hist_cal_clean_Cs137 = hist_cal_src_Cs137.Clone("Calibrated signal no BG")
hist_cal_clean_Cs137.Add(hist_cal_bg_Cs137, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv1'):
    ROOT.gROOT.FindObject('cv1').Close()
cv1 = ROOT.TCanvas("cv1", "cv1", 1600, 800)
cv1.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(1)
hist_cal_bg_Cs137.SetLineColor(colors[0])
hist_cal_src_Cs137.SetLineColor(colors[1])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Cs137, "Background", "l")
lg1.AddEntry(hist_cal_src_Cs137, r"Signal ^{137}Cs", "l")
hist_cal_src_Cs137.Draw("hist")
hist_cal_bg_Cs137.Draw("sames hist")
lg1.Draw()
cv1.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv1.cd(2)
hist_cal_clean_Cs137.SetLineColor(colors[2])
hist_cal_src_Cs137.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Cs137.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Cs137.Draw("hist")
lg2.AddEntry(hist_cal_clean_Cs137, r"^{137}Cs bg subtracted", "l")
lg2.Draw()
cv1.cd(2).SetLogy()

cv1.Draw()

#Calculate energy resolution
res_Cs137 = energy_resolution(hist_cal_clean_Cs137, [(0.3, 0.9)], [Cs137_MeV])
print(res_Cs137)



[4.673286330430341]
****************************************
         Invalid FitResult  (status = 4 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      120.727
NDf                       =          116
Edm                       =  0.000200532
NCalls                    =         1352
Constant                  =      871.918   +/-   2379.28     
Mean                      =     -5.24115   +/-   2.78506     
Sigma                     =      1.31368   +/-   0.508412     	 (limited)


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <Fit>: Abnormal termination of minimization.


In [12]:
#Calibration based on the 3 peaks of the Na-22 spectrum
hist = hists['Na-22_clean']
energy_ranges = [(90, 120), (200, 300), (300, 400)]
energies = Na22_MeV

c_fit = calibration_fit(hist, energy_ranges, energies)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Na22 = calibrated_histogram(c_fit, bg_acqs[0], BITS12)
hist_cal_bg_Na22.Scale(1/bg_acqs[0].exposure) 
hist_cal_src_Na22 = calibrated_histogram(c_fit, src_acqs[0], BITS12)
hist_cal_src_Na22.Scale(1/src_acqs[0].exposure) 
hist_cal_clean_Na22 = hist_cal_src_Na22.Clone("Calibrated signal no BG")
hist_cal_clean_Na22.Add(hist_cal_bg_Na22, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv2'):
    ROOT.gROOT.FindObject('cv2').Close()
cv2 = ROOT.TCanvas("cv2", "cv2", 1600, 800)
cv2.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(1)
hist_cal_bg_Na22.SetLineColor(colors[0])
hist_cal_src_Na22.SetLineColor(colors[1])
hist_cal_src_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_src_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
lg1.AddEntry(hist_cal_bg_Na22, "Background", "l")
lg1.AddEntry(hist_cal_src_Na22, r"Signal ^{22}Na", "l")
hist_cal_src_Na22.Draw("hist")
hist_cal_bg_Na22.Draw("sames hist")
lg1.Draw()
cv2.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv2.cd(2)
hist_cal_clean_Na22.SetLineColor(colors[2])
hist_cal_clean_Na22.GetXaxis().SetTitle("Energy [MeV]")
hist_cal_clean_Na22.GetYaxis().SetTitle(r"Count rate [Hz]")
hist_cal_clean_Na22.Draw("hist")
lg2.AddEntry(hist_cal_clean_Na22, r"^{22}Na bg subtracted", "l")
lg2.Draw()
cv2.cd(2).SetLogy()

cv2.Draw()

#Calculate energy resolution
res_Na22 = energy_resolution(hist_cal_clean_Na22, [(0.13, 0.6), (1.1, 1.5), (1.5 , 2.1)],  Na22_MeV)
print(res_Na22)



[0.5212306578445631, 0.20483011604488724, 0.17000514353095905]
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      13.2742
NDf                       =           27
Edm                       =  6.82104e-08
NCalls                    =          115
Constant                  =      1220.98   +/-   10.6387     
Mean                      =      110.016   +/-   0.473656    
Sigma                     =      19.7974   +/-   0.794329     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      182.187
NDf                       =           97
Edm                       =  1.14058e-05
NCalls                    =           62
Constant                  =      331.867   +/-   4.2191      
Mean                      =      259.724   +/-   0.285473    
Sigma                     =      22.6716   +/-   0.325946     	 (limited)
****************************************
Minimizer is Minuit2 / Migrad


Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: h_cal (Potential memory leak).


In [ ]:
#Calibration based on the 3 peaks of the Na-22 spectrum
hist = hists['Am-241_clean']
energy_ranges = [(0, 1), (5, 30)]
energies = [0, Am241_MeV]

c_fit = calibration_fit(hist, energy_ranges, energies)
#print(c_fit)

#Calibrating Na22 histograms based on Na22 calibration fit
hist_cal_bg_Am241  = calibrated_histogram(c_fit, acqs[8], BITS12)
hist_cal_bg_Am241.Scale(1/acqs[8].exposure) 
hist_cal_src_Am241 = calibrated_histogram(c_fit, acqs[7], BITS12)
hist_cal_src_Am241.Scale(1/acqs[7].exposure) 
hist_cal_clean_Am241 = hist_cal_src_Am241.Clone("Calibrated signal no BG")
hist_cal_clean_Am241.Add(hist_cal_bg_Am241, -1)


# Plot the Cs-137 histograms
if ROOT.gROOT.FindObject('cv4'):
    ROOT.gROOT.FindObject('cv4').Close()
cv4 = ROOT.TCanvas("cv4", "cv4", 1600, 800)
cv4.Divide(2,1)

lg1 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv4.cd(1)
hist_cal_bg_Am241.SetLineColor(colors[0])
hist_cal_src_Am241.SetLineColor(colors[1])
lg1.AddEntry(hist_cal_bg_Am241, "Background", "l")
lg1.AddEntry(hist_cal_src_Am241, r"Signal ^{241}Am", "l")
hist_cal_src_Am241.Draw("hist")
hist_cal_bg_Am241.Draw("sames hist")
lg1.Draw()
cv4.cd(1).SetLogy()
#hist_cal_bg.GetYaxis().SetRangeUser(0,1e4)


lg2 = ROOT.TLegend(0.48, 0.71, 0.75, 0.83)
cv4.cd(2)
hist_cal_clean_Am241.SetLineColor(colors[2])
hist_cal_clean_Am241.Draw("hist")
lg2.AddEntry(hist_cal_clean_Am241, r"^{241}Am bg subtracted", "l")
lg2.Draw()
cv4.cd(2).SetLogy()

cv4.Draw()

#Calculate energy resolution
res_Am241 = energy_resolution(hist_cal_clean_Am241, [(0.01, 0.09)],  [Am241_MeV])
print(res_Am241)

